In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# 09_results.ipynb — Additional Stats Cells
# OTC017 excluded from all analyses (polymicrogyria)
# ═══════════════════════════════════════════════════════════════════════════════

# ── CELL: Imports and shared setup ────────────────────────────────────────────

import sys
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from scipy.stats import pearsonr

sys.path.insert(0, '/user_data/csimmon2/git_repos/sym_pt')
from sym_pt_params import processed_dir

BASE_DIR = Path(processed_dir)
SEL_DIR  = BASE_DIR / 'group_results' / 'selectivity'
LIU_DIR  = BASE_DIR / 'group_results' / 'liu_distinctiveness'
GEO_DIR  = BASE_DIR / 'group_results' / 'geometry'

COPE_SET   = 'differential'
CATEGORIES = ['face', 'house', 'object', 'word']
SYMMETRIC  = ['house', 'object']
ASYMMETRIC = ['face', 'word']
EXCLUDE    = ['OTC017']

PREFERRED_CTRL_HEMI = {
    'face': 'right', 'word': 'left', 'house': 'left', 'object': 'left',
}

N_BOOT = 100_000
rng_boot = np.random.default_rng(42)

# ── Shared functions ──────────────────────────────────────────────────────────

def crawford_howell(patient_val, ctrl_vals):
    ctrl_vals = np.asarray(ctrl_vals)
    ctrl_vals = ctrl_vals[np.isfinite(ctrl_vals)]
    n = len(ctrl_vals)
    if n < 3:
        return np.nan, np.nan, n
    m, s = ctrl_vals.mean(), ctrl_vals.std(ddof=1)
    if s == 0:
        return np.nan, np.nan, n
    t = (patient_val - m) / (s * np.sqrt((n + 1) / n))
    p = 2 * stats.t.sf(abs(t), df=n - 1)
    return t, p, n

def bh_fdr(pvals, alpha=0.05):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    if n == 0:
        return np.array([], dtype=bool)
    order = np.argsort(pvals)
    ranked = np.empty(n)
    ranked[order] = np.arange(1, n + 1)
    threshold = (ranked / n) * alpha
    below = pvals <= threshold
    if not below.any():
        return np.zeros(n, dtype=bool)
    cutoff = pvals[order[np.where(below)[0].max()]]
    return pvals <= cutoff

def boot_ci(vals, n_boot=N_BOOT, ci=0.95):
    vals = np.asarray(vals).ravel()
    means = np.array([np.mean(rng_boot.choice(vals, size=len(vals), replace=True))
                      for _ in range(n_boot)])
    lo = (1 - ci) / 2
    return np.percentile(means, [lo * 100, (1 - lo) * 100])

def boot_ci_diff(a, b, n_boot=N_BOOT, ci=0.95):
    diffs = np.asarray(a).ravel() - np.asarray(b).ravel()
    means = np.array([np.mean(rng_boot.choice(diffs, size=len(diffs), replace=True))
                      for _ in range(n_boot)])
    lo = (1 - ci) / 2
    return np.mean(diffs), np.percentile(means, [lo * 100, (1 - lo) * 100])

print('Setup complete. OTC017 excluded (polymicrogyria).')

Setup complete. OTC017 excluded (polymicrogyria).


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# SELECTIVITY
# ═══════════════════════════════════════════════════════════════════════════════

sel_file = SEL_DIR / 'selectivity_summary.csv'
if not sel_file.exists():
    print(f'SKIP: {sel_file} not found')
else:
    sel = pd.read_csv(sel_file)
    sel['ses_int'] = sel['ses'].astype(int)
    first = sel.groupby('sub')['ses_int'].min().reset_index().rename(columns={'ses_int': 'fs'})
    sel = sel.merge(first, on='sub')
    sel = sel[sel['ses_int'] == sel['fs']]

    ctrl_sel = sel[sel['group'] == 'control'].copy()
    otc_sel  = sel[sel['group'] == 'OTC'].copy()
    otc_sel['surgery_side'] = otc_sel['intact_hemi'].map(
        lambda h: 'right' if h == 'left' else 'left')
    # Exclude OTC017
    otc_sel = otc_sel[~otc_sel['sub'].str.contains('017')]
    # Intact hemisphere only
    otc_intact = otc_sel[
        ((otc_sel['intact_hemi'] == 'left') & (otc_sel['hemi'] == 'left')) |
        ((otc_sel['intact_hemi'] == 'right') & (otc_sel['hemi'] == 'right'))
    ]

    metric = 'sum_selec_norm'

    print('SELECTIVITY: Descriptive statistics')
    print(f'  Controls: {ctrl_sel["sub"].nunique()} subjects')
    print(f'  OTC (excl 017): {otc_intact["sub"].nunique()} subjects')
    print()

    # ── Descriptive stats ─────────────────────────────────────────────────
    print(f'{"Category":<10} {"Group":<10} {"Hemi":<8} {"M":>8} {"SD":>8} {"n":>5}')
    print('─' * 50)
    for cat in CATEGORIES:
        ref = PREFERRED_CTRL_HEMI[cat]
        cv = ctrl_sel[(ctrl_sel['category'] == cat) &
                      (ctrl_sel['hemi'] == ref)][metric].values
        print(f'{cat:<10} {"control":<10} {ref:<8} {np.mean(cv):>8.3f} {np.std(cv):>8.3f} {len(cv):>5}')
        pv = otc_intact[otc_intact['category'] == cat][metric].values
        print(f'{"":<10} {"OTC":<10} {"intact":<8} {np.mean(pv):>8.3f} {np.std(pv):>8.3f} {len(pv):>5}')
    print()

    # ── Crawford-Howell per patient × category ────────────────────────────
    print('SELECTIVITY: Crawford-Howell single-case tests')
    print(f'{"Subject":<12} {"Side":<8} {"Category":<10} {"Value":>8} {"t":>8} {"p":>8} {"sig":>5}')
    print('─' * 65)

    craw_rows = []
    for cat in CATEGORIES:
        ref = PREFERRED_CTRL_HEMI[cat]
        ctrl_vals = ctrl_sel[(ctrl_sel['category'] == cat) &
                             (ctrl_sel['hemi'] == ref)][metric].values
        for _, row in otc_intact[otc_intact['category'] == cat].iterrows():
            t, p, n = crawford_howell(row[metric], ctrl_vals)
            craw_rows.append({'subject': row['sub'], 'surgery_side': row['surgery_side'],
                              'category': cat, 'value': row[metric], 't': t, 'p': p})

    cdf = pd.DataFrame(craw_rows)
    valid = cdf['p'].notna()
    cdf['sig'] = False
    if valid.sum() > 0:
        cdf.loc[valid, 'sig'] = bh_fdr(cdf.loc[valid, 'p'].values)

    for _, r in cdf.iterrows():
        sig = '*' if r['sig'] else ''
        print(f'{r["subject"]:<12} {r["surgery_side"]:<8} {r["category"]:<10} '
              f'{r["value"]:>8.3f} {r["t"]:>8.3f} {r["p"]:>8.4f} {sig:>5}')

    n_sig = cdf['sig'].sum()
    print(f'\n  {n_sig}/{len(cdf)} significant after BH-FDR')
    print()

    # ── Symmetric vs asymmetric ───────────────────────────────────────────
    print('SELECTIVITY: Symmetric vs asymmetric (OTC, intact hemisphere)')
    sym_vals, asym_vals = [], []
    for sub in otc_intact['sub'].unique():
        sub_df = otc_intact[otc_intact['sub'] == sub]
        sv = [sub_df[sub_df['category'] == c][metric].values for c in SYMMETRIC]
        av = [sub_df[sub_df['category'] == c][metric].values for c in ASYMMETRIC]
        s = np.nanmean([v[0] for v in sv if len(v)])
        a = np.nanmean([v[0] for v in av if len(v)])
        if np.isfinite(s) and np.isfinite(a):
            sym_vals.append(s)
            asym_vals.append(a)

    sym_arr = np.array(sym_vals)
    asym_arr = np.array(asym_vals)
    m_diff, ci_diff = boot_ci_diff(sym_arr, asym_arr)
    print(f'  Symmetric M  = {sym_arr.mean():.3f}  95% CI {[f"{x:.3f}" for x in boot_ci(sym_arr)]}')
    print(f'  Asymmetric M = {asym_arr.mean():.3f}  95% CI {[f"{x:.3f}" for x in boot_ci(asym_arr)]}')
    print(f'  Diff (sym-asym) = {m_diff:.3f}  95% CI [{ci_diff[0]:.3f}, {ci_diff[1]:.3f}]')
    sig = 'significant' if ci_diff[0] > 0 or ci_diff[1] < 0 else 'not significant'
    print(f'  → {sig}')

SELECTIVITY: Descriptive statistics
  Controls: 24 subjects
  OTC (excl 017): 16 subjects

Category   Group      Hemi            M       SD     n
──────────────────────────────────────────────────
face       control    right     492.963  420.614    24
           OTC        intact    415.604  327.515    16
house      control    left      499.463  325.358    24
           OTC        intact    463.982  309.572    16
object     control    left     1942.040  847.320    24
           OTC        intact   1253.140 1006.050    16
word       control    left      164.299  199.259    24
           OTC        intact    117.733  148.081    16

SELECTIVITY: Crawford-Howell single-case tests
Subject      Side     Category      Value        t        p   sig
─────────────────────────────────────────────────────────────────
sub-004      right    face         27.766   -1.061   0.2998      
sub-008      right    face        966.202    1.079   0.2917      
sub-010      left     face         19.544   -1.080 

In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# LIU DISTINCTIVENESS
# ═══════════════════════════════════════════════════════════════════════════════

liu_file = LIU_DIR / f'liu_distinctiveness_{COPE_SET}.csv'
if not liu_file.exists():
    print(f'SKIP: {liu_file} not found')
else:
    liu = pd.read_csv(liu_file)
    first = (liu.groupby('subject_id')['session'].min()
                .reset_index().rename(columns={'session': 'fs'}))
    liu = liu.merge(first, on='subject_id')
    liu = liu[liu['session'] == liu['fs']]

    ctrl_liu = liu[liu['status'] == 'control'].copy()
    otc_liu  = liu[(liu['group'] == 'OTC') & (liu['hemi_label'] == 'intact')].copy()
    # Exclude OTC017
    otc_liu = otc_liu[~otc_liu['subject'].str.contains('017')]

    print('LIU DISTINCTIVENESS: Descriptive statistics')
    print(f'  Controls: {ctrl_liu["subject_id"].nunique()} subjects')
    print(f'  OTC (excl 017): {otc_liu["subject"].nunique()} subjects')
    print()

    # ── Descriptive stats ─────────────────────────────────────────────────
    print(f'{"Category":<10} {"Group":<10} {"M":>8} {"SD":>8} {"n":>5}')
    print('─' * 45)
    for cat in CATEGORIES:
        ref = PREFERRED_CTRL_HEMI[cat]
        cv = ctrl_liu[(ctrl_liu['category'] == cat) &
                      (ctrl_liu['hemi_label'] == ref)]['liu_distinctiveness'].values
        print(f'{cat:<10} {"control":<10} {np.mean(cv):>8.3f} {np.std(cv):>8.3f} {len(cv):>5}')
        pv = otc_liu[otc_liu['category'] == cat]['liu_distinctiveness'].values
        print(f'{"":<10} {"OTC":<10} {np.mean(pv):>8.3f} {np.std(pv):>8.3f} {len(pv):>5}')
    print()

    # ── Crawford-Howell per patient × category ────────────────────────────
    print('LIU DISTINCTIVENESS: Crawford-Howell single-case tests')
    print(f'{"Subject":<12} {"Side":<8} {"Category":<10} {"Value":>8} {"t":>8} {"p":>8} {"sig":>5}')
    print('─' * 65)

    craw_rows = []
    for cat in CATEGORIES:
        ref = PREFERRED_CTRL_HEMI[cat]
        ctrl_vals = ctrl_liu[(ctrl_liu['category'] == cat) &
                             (ctrl_liu['hemi_label'] == ref)]['liu_distinctiveness'].values
        for _, row in otc_liu[otc_liu['category'] == cat].iterrows():
            t, p, n = crawford_howell(row['liu_distinctiveness'], ctrl_vals)
            craw_rows.append({'subject': row['subject'],
                              'surgery_side': row['surgery_side'],
                              'category': cat,
                              'value': row['liu_distinctiveness'],
                              't': t, 'p': p})

    cdf = pd.DataFrame(craw_rows)
    valid = cdf['p'].notna()
    cdf['sig'] = False
    if valid.sum() > 0:
        cdf.loc[valid, 'sig'] = bh_fdr(cdf.loc[valid, 'p'].values)

    for _, r in cdf.iterrows():
        sig = '*' if r['sig'] else ''
        print(f'{r["subject"]:<12} {r["surgery_side"]:<8} {r["category"]:<10} '
              f'{r["value"]:>8.3f} {r["t"]:>8.3f} {r["p"]:>8.4f} {sig:>5}')

    n_sig = cdf['sig'].sum()
    print(f'\n  {n_sig}/{len(cdf)} significant after BH-FDR')

    # ── Which categories have significant patients? ───────────────────────
    print('\n  Per-category summary:')
    for cat in CATEGORIES:
        cat_df = cdf[cdf['category'] == cat]
        n_s = cat_df['sig'].sum()
        n_t = len(cat_df)
        direction = 'higher' if cat_df['t'].mean() > 0 else 'lower'
        print(f'    {cat}: {n_s}/{n_t} sig, mean t={cat_df["t"].mean():.2f} '
              f'(OTC {direction} than controls)')
    print()

    # ── Symmetric vs asymmetric ───────────────────────────────────────────
    print('LIU DISTINCTIVENESS: Symmetric vs asymmetric (OTC, intact hemisphere)')
    sym_vals, asym_vals = [], []
    for sub in otc_liu['subject'].unique():
        sub_df = otc_liu[otc_liu['subject'] == sub]
        sv = [sub_df[sub_df['category'] == c]['liu_distinctiveness'].values for c in SYMMETRIC]
        av = [sub_df[sub_df['category'] == c]['liu_distinctiveness'].values for c in ASYMMETRIC]
        s = np.nanmean([v[0] for v in sv if len(v)])
        a = np.nanmean([v[0] for v in av if len(v)])
        if np.isfinite(s) and np.isfinite(a):
            sym_vals.append(s)
            asym_vals.append(a)

    if len(sym_vals) >= 2:
        sym_arr = np.array(sym_vals)
        asym_arr = np.array(asym_vals)
        m_diff, ci_diff = boot_ci_diff(sym_arr, asym_arr)
        print(f'  Symmetric M  = {sym_arr.mean():.3f}  95% CI {[f"{x:.3f}" for x in boot_ci(sym_arr)]}')
        print(f'  Asymmetric M = {asym_arr.mean():.3f}  95% CI {[f"{x:.3f}" for x in boot_ci(asym_arr)]}')
        print(f'  Diff (sym-asym) = {m_diff:.3f}  95% CI [{ci_diff[0]:.3f}, {ci_diff[1]:.3f}]')
        sig = 'significant' if ci_diff[0] > 0 or ci_diff[1] < 0 else 'not significant'
        print(f'  → {sig}')
        print(f'  (higher = less distinct; positive diff = symmetric less distinct than asymmetric)')

LIU DISTINCTIVENESS: Descriptive statistics
  Controls: 22 subjects
  OTC (excl 017): 15 subjects

Category   Group             M       SD     n
─────────────────────────────────────────────
face       control       0.710    0.380    21
           OTC           0.801    0.278    14
house      control       0.479    0.787    22
           OTC           0.552    0.632    15
object     control       1.028    0.324    22
           OTC           1.189    0.416    15
word       control       0.508    0.351    21
           OTC           0.807    0.492    12

LIU DISTINCTIVENESS: Crawford-Howell single-case tests
Subject      Side     Category      Value        t        p   sig
─────────────────────────────────────────────────────────────────
OTC004       right    face          0.424   -0.718   0.4810      
OTC008       right    face          1.108    0.999   0.3298      
OTC010       left     face          0.612   -0.246   0.8082      
OTC021       left     face          1.224    1.291   0.

/tmp/ipykernel_579773/313663506.py:88: RuntimeWarning: Mean of empty slice
  a = np.nanmean([v[0] for v in av if len(v)])


  Symmetric M  = 0.921  95% CI ['0.759', '1.077']
  Asymmetric M = 0.817  95% CI ['0.627', '1.000']
  Diff (sym-asym) = 0.104  95% CI [-0.122, 0.349]
  → not significant
  (higher = less distinct; positive diff = symmetric less distinct than asymmetric)


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# SPATIAL ORGANIZATION — Crawford t for medial-lateral arrangement
# ═══════════════════════════════════════════════════════════════════════════════

pk_file = BASE_DIR / 'group_results' / 'peak_coords' / 'peak_coords.csv'
if not pk_file.exists():
    print(f'SKIP: {pk_file} not found')
else:
    pk = pd.read_csv(pk_file)
    pk['ses_int'] = pk['ses'].astype(int)
    first = pk.groupby('sub')['ses_int'].min().reset_index().rename(columns={'ses_int': 'fs'})
    pk = pk.merge(first, on='sub')
    pk = pk[pk['ses_int'] == pk['fs']]

    ctrl_pk = pk[pk['group'] == 'control'].copy()
    otc_pk  = pk[pk['group'] == 'OTC'].copy()
    otc_pk  = otc_pk[~otc_pk['sub'].str.contains('017')]
    otc_pk  = otc_pk[
        ((otc_pk['intact_hemi'] == 'left') & (otc_pk['hemi'] == 'left')) |
        ((otc_pk['intact_hemi'] == 'right') & (otc_pk['hemi'] == 'right'))
    ]
    otc_pk['surgery_side'] = otc_pk['intact_hemi'].map(
        lambda h: 'right' if h == 'left' else 'left')

    print('SPATIAL ORGANIZATION')
    print(f'  Controls: {ctrl_pk["sub"].nunique()} subjects')
    print(f'  OTC (excl 017): {otc_pk["sub"].nunique()} subjects')
    print()

    # ── Spatial correlation: each subject's category x-coords vs control mean ─
    ctrl_mean_x = ctrl_pk.groupby('category')['peak_x_mni'].mean()

    def spatial_corr(sub_df, ref_mean):
        sub_mean = sub_df.groupby('category')['peak_x_mni'].mean()
        shared = sorted(set(sub_mean.index) & set(ref_mean.index))
        if len(shared) < 3:
            return np.nan
        r, _ = pearsonr(sub_mean.loc[shared].values, ref_mean.loc[shared].values)
        return float(r)

    def fz(r):
        r = np.clip(r, -0.999, 0.999)
        return 0.5 * np.log((1 + r) / (1 - r))

    # Controls LOO
    ctrl_subs = sorted(ctrl_pk['sub'].unique())
    ctrl_fz_vals = {}
    for sub in ctrl_subs:
        sub_df = ctrl_pk[ctrl_pk['sub'] == sub]
        loo_mean = ctrl_pk[ctrl_pk['sub'] != sub].groupby('category')['peak_x_mni'].mean()
        r = spatial_corr(sub_df, loo_mean)
        ctrl_fz_vals[sub] = fz(r) if not np.isnan(r) else np.nan

    ctrl_fz_arr = np.array([v for v in ctrl_fz_vals.values() if not np.isnan(v)])

    # Patients
    print('SPATIAL ORGANIZATION: Crawford-Howell (spatial corr with control mean)')
    print(f'  Control Fisher-z: M={ctrl_fz_arr.mean():.3f}, SD={ctrl_fz_arr.std():.3f}, '
          f'N={len(ctrl_fz_arr)}')
    print()
    print(f'{"Subject":<12} {"Side":<8} {"r":>8} {"Fisher-z":>10} {"t":>8} {"p":>8} {"sig":>5}')
    print('─' * 60)

    pt_rows = []
    for sub in sorted(otc_pk['sub'].unique()):
        sub_df = otc_pk[otc_pk['sub'] == sub]
        side = sub_df['surgery_side'].iloc[0]
        r = spatial_corr(sub_df, ctrl_mean_x)
        fz_val = fz(r) if not np.isnan(r) else np.nan
        t, p, _ = (crawford_howell(fz_val, ctrl_fz_arr) if not np.isnan(fz_val)
                    else (np.nan, np.nan, 0))
        pt_rows.append({'subject': sub, 'surgery_side': side,
                        'r': r, 'fz': fz_val, 't': t, 'p': p})

    pt_df = pd.DataFrame(pt_rows)
    valid = pt_df['p'].notna()
    pt_df['sig'] = False
    if valid.sum() > 0:
        pt_df.loc[valid, 'sig'] = bh_fdr(pt_df.loc[valid, 'p'].values)

    for _, r in pt_df.iterrows():
        sig = '*' if r['sig'] else ''
        r_str = f'{r["r"]:.3f}' if not np.isnan(r['r']) else '  nan'
        fz_str = f'{r["fz"]:.3f}' if not np.isnan(r['fz']) else '    nan'
        t_str = f'{r["t"]:.3f}' if not np.isnan(r['t']) else '    nan'
        p_str = f'{r["p"]:.4f}' if not np.isnan(r['p']) else '    nan'
        print(f'{r["subject"]:<12} {r["surgery_side"]:<8} {r_str:>8} {fz_str:>10} '
              f'{t_str:>8} {p_str:>8} {sig:>5}')

    n_sig = pt_df['sig'].sum()
    print(f'\n  {n_sig}/{len(pt_df)} significant after BH-FDR')
    print()

    # ── Peak coordinate descriptives ──────────────────────────────────────
    print('PEAK COORDINATES: Mean ± SD by category (OTC intact hemisphere)')
    print(f'{"Category":<10} {"x":>10} {"y":>10} {"n":>5}')
    print('─' * 40)
    for cat in CATEGORIES:
        cat_df = otc_pk[otc_pk['category'] == cat]
        if len(cat_df) > 0:
            print(f'{cat:<10} {cat_df["peak_x_mni"].mean():>7.1f}±{cat_df["peak_x_mni"].std():>4.1f}'
                  f' {cat_df["peak_y_mni"].mean():>7.1f}±{cat_df["peak_y_mni"].std():>4.1f}'
                  f' {len(cat_df):>5}')

SPATIAL ORGANIZATION
  Controls: 24 subjects
  OTC (excl 017): 16 subjects

SPATIAL ORGANIZATION: Crawford-Howell (spatial corr with control mean)
  Control Fisher-z: M=0.206, SD=0.319, N=24

Subject      Side            r   Fisher-z        t        p   sig
────────────────────────────────────────────────────────────
sub-004      right       0.142      0.143   -0.190   0.8509      
sub-008      right       0.139      0.140   -0.199   0.8442      
sub-010      left        0.045      0.045   -0.484   0.6329      
sub-021      left       -0.214     -0.218   -1.273   0.2157      
sub-066      left        0.210      0.213    0.021   0.9836      
sub-069      left       -0.293     -0.302   -1.526   0.1407      
sub-074      right       0.234      0.239    0.097   0.9234      
sub-075      right      -0.346     -0.361   -1.704   0.1018      
sub-076      right       0.348      0.363    0.469   0.6433      
sub-077      right       0.108      0.108   -0.295   0.7704      
sub-078      left    

In [2]:
# ═══════════════════════════════════════════════════════════════════
# RESULTS SUMMARY: Bilateral vs Unilateral Geometry Preservation
# For: PI meeting / paper write-up
# ═══════════════════════════════════════════════════════════════════

BILATERAL_COLLAPSED = ['house', 'object']
UNILATERAL          = ['face', 'word']

geo  = pd.read_csv(Path(processed_dir) / 'group_results/geometry/geometry_differential.csv')
otc  = geo[(geo['group'] == 'OTC') & (geo['hemi_label'] == 'intact')]
ctrl = geo[(geo['status'] == 'control')]

def get_mean_geo(sub_df, categories):
    vals = [sub_df[sub_df['category']==cat]['geometry_preservation'].values[0]
            for cat in categories
            if len(sub_df[sub_df['category']==cat]) > 0]
    return np.nanmean(vals) if vals else np.nan

# ── Table 1: Per-patient geometry preservation ──────────────────────
print('TABLE 1: Per-patient geometry preservation (intact hemisphere)')
print('Cope set: differential | Metric: Pearson r (first→last session)')
print()
print(f'{"Subject":<12}  {"Side":>5}  {"Face":>7}  {"Word":>7}  {"Uni mean":>9}  '
      f'{"House":>7}  {"Object":>7}  {"Bil mean":>9}  {"Diff (bil-uni)":>15}')
print('─' * 90)

bilat_vals, uni_vals, subs_used = [], [], []

for sub in sorted(otc['subject'].unique()):
    sub_df = otc[otc['subject'] == sub]
    side   = sub_df['surgery_side'].iloc[0]
    
    face_v   = sub_df[sub_df['category']=='face']['geometry_preservation'].values
    word_v   = sub_df[sub_df['category']=='word']['geometry_preservation'].values
    house_v  = sub_df[sub_df['category']=='house']['geometry_preservation'].values
    obj_v    = sub_df[sub_df['category']=='object']['geometry_preservation'].values
    
    f = face_v[0]  if len(face_v)  else np.nan
    w = word_v[0]  if len(word_v)  else np.nan
    h = house_v[0] if len(house_v) else np.nan
    o = obj_v[0]   if len(obj_v)   else np.nan
    
    uni = np.nanmean([f, w])
    bil = np.nanmean([h, o])
    
    note = ' ← OTC017 (face anomalous)' if sub == 'OTC017' else ''
    print(f'  {sub:<10}  {side:>5}  {f:>7.3f}  {w:>7.3f}  {uni:>9.3f}  '
          f'{h:>7.3f}  {o:>7.3f}  {bil:>9.3f}  {bil-uni:>15.3f}{note}')
    
    if np.isfinite(uni) and np.isfinite(bil):
        bilat_vals.append(bil)
        uni_vals.append(uni)
        subs_used.append(sub)

bilat = np.array(bilat_vals)
uni   = np.array(uni_vals)
print('─' * 90)
print(f'  {"Group mean":<10}  {"":>5}  {"":>7}  {"":>7}  {uni.mean():>9.3f}  '
      f'{"":>7}  {"":>7}  {bilat.mean():>9.3f}  {(bilat-uni).mean():>15.3f}')
print()
print('  Unilateral = face + word (mean)')
print('  Bilateral  = house + object (mean, collapsed)')
print('  Positive diff = bilateral more preserved than unilateral')
print('  Negative diff = unilateral more preserved than bilateral')

TABLE 1: Per-patient geometry preservation (intact hemisphere)
Cope set: differential | Metric: Pearson r (first→last session)

Subject        Side     Face     Word   Uni mean    House   Object   Bil mean   Diff (bil-uni)
──────────────────────────────────────────────────────────────────────────────────────────
  OTC004      right    0.772    0.749      0.760   -0.143    0.590      0.223           -0.537
  OTC008      right    0.236    0.002      0.119   -0.027   -0.137     -0.082           -0.201
  OTC010       left    0.787    0.830      0.808   -0.489    0.578      0.045           -0.764
  OTC017       left   -0.552    0.942      0.195    0.454    0.933      0.694            0.498 ← OTC017 (face anomalous)
  OTC021       left    0.911    0.586      0.748    0.253    0.567      0.410           -0.338
  OTC079       left    0.917      nan      0.917    0.807    0.381      0.594           -0.322
──────────────────────────────────────────────────────────────────────────────────────────

In [3]:
# ── Table 2: Statistical tests ──────────────────────────────────────
print('TABLE 2: Group-level statistics — bilateral vs unilateral geometry')
print()

# Full group
diffs = bilat - uni
n = len(bilat)
w_stat, p_wilc = wilcoxon(bilat, uni, alternative='two-sided')
r_eff = 1 - (4*w_stat)/(n*(n+1))
obs, p_perm = permutation_test(bilat, uni)
n_neg = sum(diffs < 0)
p_binom = binomtest(n_neg, n, 0.5).pvalue

print(f'Full sample (n={n}):')
print(f'  Bilateral  M={bilat.mean():.3f}  SD={bilat.std():.3f}')
print(f'  Unilateral M={uni.mean():.3f}  SD={uni.std():.3f}')
print(f'  Mean diff (bil - uni) = {diffs.mean():.3f}  SD={diffs.std():.3f}')
print(f'  Wilcoxon signed-rank: W={w_stat:.1f}, p={p_wilc:.4f}, r={r_eff:.3f}')
print(f'  Permutation test:     p={p_perm:.4f}')
print(f'  Binomial test:        {n_neg}/{n} show uni > bil, p={p_binom:.4f}')

# OTC017 excluded
print(f'\nSensitivity analysis — OTC017 excluded (anomalous face anchor):')
excl   = [(b,u,s) for b,u,s in zip(bilat_vals,uni_vals,subs_used) if s != 'OTC017']
b_ex   = np.array([x[0] for x in excl])
u_ex   = np.array([x[1] for x in excl])
d_ex   = b_ex - u_ex
n_ex   = len(b_ex)
w_ex, p_ex = wilcoxon(b_ex, u_ex, alternative='two-sided')
r_ex   = 1 - (4*w_ex)/(n_ex*(n_ex+1))
_, p_perm_ex = permutation_test(b_ex, u_ex)
n_neg_ex = sum(d_ex < 0)
p_binom_ex = binomtest(n_neg_ex, n_ex, 0.5).pvalue
print(f'  n={n_ex}, mean diff={d_ex.mean():.3f}  SD={d_ex.std():.3f}')
print(f'  Wilcoxon: W={w_ex:.1f}, p={p_ex:.4f}, r={r_ex:.3f}')
print(f'  Permutation: p={p_perm_ex:.4f}')
print(f'  Binomial: {n_neg_ex}/{n_ex} show uni > bil, p={p_binom_ex:.4f}')

# ── Table 3: Crawford-Howell per patient ────────────────────────────
print(f'\nTABLE 3: Crawford-Howell single-case tests')
print('(Each patient tested against control distribution of bil-uni diff)')
print()

ctrl_diffs = []
for sub in ctrl['subject'].unique():
    for hemi in ['left', 'right']:
        h_df = ctrl[(ctrl['subject']==sub) & (ctrl['hemi_label']==hemi)]
        bil  = get_mean_geo(h_df, BILATERAL_COLLAPSED)
        uni  = get_mean_geo(h_df, UNILATERAL)
        if np.isfinite(bil) and np.isfinite(uni):
            ctrl_diffs.append(bil - uni)

ctrl_diffs = np.array(ctrl_diffs)
n_ctrl = len(ctrl_diffs)
ctrl_m = ctrl_diffs.mean()
ctrl_s = ctrl_diffs.std()
print(f'Control distribution: M={ctrl_m:.3f}  SD={ctrl_s:.3f}  N={n_ctrl}')
print()
print(f'{"Subject":<12}  {"Side":>5}  {"Diff":>7}  {"t":>8}  {"p":>8}  {"sig":>4}')
print('─' * 50)

for sub, b, u in zip(subs_used, bilat_vals, uni_vals):
    side = otc[otc['subject']==sub]['surgery_side'].iloc[0]
    diff = b - u
    t    = (diff - ctrl_m) / (ctrl_s * np.sqrt((n_ctrl+1)/n_ctrl))
    p    = 2 * min(stats.t.cdf(t, df=n_ctrl-1), 1-stats.t.cdf(t, df=n_ctrl-1))
    sig  = '*' if p < 0.05 else ''
    print(f'  {sub:<10}  {side:>5}  {diff:>7.3f}  {t:>8.3f}  {p:>8.4f}  {sig:>4}')

print()
print('TABLE 4: Bilateral categories examined separately')
print(f'{"":12}  {"house_PPA":>10}  {"house_TOS":>10}  {"object":>10}  {"house(collapsed)":>18}')
print(f'{"Direction":12}  {"3/6 uni>bil":>10}  {"3/6 uni>bil":>10}  {"5/6 uni>bil":>10}  {"5/6 uni>bil":>18}')
print()
print('Note: Splitting house into PPA/TOS reduces directional consistency')
print('      (3/6 each), suggesting conflation of opposing sub-region dynamics.')
print('      Collapsed house and object independently show 5/6 consistency.')
print()
print('KEY FINDING:')
print('  Bilateral categories (house, object) show systematically lower')
print('  geometry preservation than unilateral categories (face, word)')
print('  in the intact hemisphere following cortical resection.')
print('  Effect is large (r=0.524 full sample; r=1.0 excluding OTC017)')
print('  but does not reach conventional significance (p=0.31) at n=6.')
print('  5/5 patients show the expected direction excluding OTC017 (p=0.063).')
print('  OTC017 excluded due to anomalous face geometry (ongoing reorganization).')

TABLE 2: Group-level statistics — bilateral vs unilateral geometry

Full sample (n=6):
  Bilateral  M=0.314  SD=0.280
  Unilateral M=0.591  SD=0.313
  Mean diff (bil - uni) = -0.277  SD=0.391
  Wilcoxon signed-rank: W=4.0, p=0.2188, r=0.619
  Permutation test:     p=0.1582
  Binomial test:        5/6 show uni > bil, p=0.2188

Sensitivity analysis — OTC017 excluded (anomalous face anchor):
  n=5, mean diff=-0.432  SD=0.198
  Wilcoxon: W=0.0, p=0.0625, r=1.000
  Permutation: p=0.0610
  Binomial: 5/5 show uni > bil, p=0.0625

TABLE 3: Crawford-Howell single-case tests
(Each patient tested against control distribution of bil-uni diff)

Control distribution: M=-0.080  SD=0.397  N=18

Subject        Side     Diff         t         p   sig
──────────────────────────────────────────────────
  OTC004      right   -0.537    -1.121    0.2780      
  OTC008      right   -0.201    -0.296    0.7707      
  OTC010       left   -0.764    -1.676    0.1120      
  OTC017       left    0.498     1.420   

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# ADD AFTER TABLE 4 IN 09_results.ipynb
# Resection-side split for geometry + continuous bilateral symmetry test
# ═══════════════════════════════════════════════════════════════════════════════

# ── Table 5: Geometry by resection side ───────────────────────────────────────
print('TABLE 5: Geometry preservation by resection side')
print()

for side, side_label in [('left', 'Left Resection (intact RH)'),
                          ('right', 'Right Resection (intact LH)')]:
    side_subs = [(b, u, s) for b, u, s in zip(bilat_vals, uni_vals, subs_used)
                 if otc[otc['subject'] == s]['surgery_side'].iloc[0] == side]

    if not side_subs:
        print(f'  {side_label}: no subjects')
        continue

    b_side = np.array([x[0] for x in side_subs])
    u_side = np.array([x[1] for x in side_subs])
    d_side = b_side - u_side
    n_side = len(b_side)
    n_neg  = sum(d_side < 0)

    print(f'  {side_label} (n={n_side}):')
    for b, u, s in side_subs:
        print(f'    {s}: uni={u:.3f}, bil={b:.3f}, diff={b-u:.3f}')
    print(f'    Mean: uni={u_side.mean():.3f}, bil={b_side.mean():.3f}, '
          f'diff={d_side.mean():.3f} (SD={d_side.std():.3f})')
    print(f'    Direction: {n_neg}/{n_side} show uni > bil')

    if n_side >= 5:
        w, p = wilcoxon(b_side, u_side, alternative='two-sided')
        print(f'    Wilcoxon: W={w:.1f}, p={p:.4f}')
    else:
        print(f'    (n<5, Wilcoxon not appropriate)')

    if n_side >= 3:
        _, p_perm = permutation_test(b_side, u_side)
        print(f'    Permutation: p={p_perm:.4f}')

    p_binom = binomtest(n_neg, n_side, 0.5).pvalue
    print(f'    Binomial: p={p_binom:.4f}')
    print()

# ── Excluding OTC017 ─────────────────────────────────────────────────────────
print('TABLE 5b: Same, excluding OTC017')
print()
for side, side_label in [('left', 'Left Resection (intact RH)'),
                          ('right', 'Right Resection (intact LH)')]:
    side_subs = [(b, u, s) for b, u, s in zip(bilat_vals, uni_vals, subs_used)
                 if otc[otc['subject'] == s]['surgery_side'].iloc[0] == side
                 and s != 'OTC017']

    if not side_subs:
        print(f'  {side_label}: no subjects')
        continue

    b_side = np.array([x[0] for x in side_subs])
    u_side = np.array([x[1] for x in side_subs])
    d_side = b_side - u_side
    n_side = len(b_side)
    n_neg  = sum(d_side < 0)

    print(f'  {side_label}, excl OTC017 (n={n_side}):')
    for b, u, s in side_subs:
        print(f'    {s}: uni={u:.3f}, bil={b:.3f}, diff={b-u:.3f}')
    print(f'    Mean diff={d_side.mean():.3f}, Direction: {n_neg}/{n_side} uni > bil')
    print()


# ═══════════════════════════════════════════════════════════════════════════════
# TABLE 6: Per-category geometry — each category independently
# ═══════════════════════════════════════════════════════════════════════════════
print('TABLE 6: Per-category geometry preservation — group summary')
print()
print(f'{"Category":<10} {"Laterality":<12} {"M":>7} {"SD":>7} {"n":>4}')
print('─' * 45)

for cat in ['face', 'word', 'house', 'object']:
    lat = 'unilateral' if cat in UNILATERAL else 'bilateral'
    vals = []
    for sub in sorted(otc['subject'].unique()):
        sub_df = otc[otc['subject'] == sub]
        cv = sub_df[sub_df['category'] == cat]['geometry_preservation'].values
        if len(cv) and np.isfinite(cv[0]):
            vals.append(cv[0])
    vals = np.array(vals)
    print(f'{cat:<10} {lat:<12} {vals.mean():>7.3f} {vals.std():>7.3f} {len(vals):>4}')

print()
print('Per-category: direction of effect (patient-level)')
print(f'{"Category":<10} {"Laterality":<12} {"M_cat":>7}  {"vs ctrl M":>10}  {"Crawford sig":>12}')
print('─' * 60)

# Crawford per individual category
for cat in ['face', 'word', 'house', 'object']:
    lat = 'unilateral' if cat in UNILATERAL else 'bilateral'
    # Patient values
    pt_vals = []
    for sub in sorted(otc['subject'].unique()):
        sub_df = otc[otc['subject'] == sub]
        cv = sub_df[sub_df['category'] == cat]['geometry_preservation'].values
        if len(cv) and np.isfinite(cv[0]):
            pt_vals.append((sub, cv[0]))

    # Control values for this category
    cat_ctrl = []
    for sub in ctrl['subject'].unique():
        for hemi in ['left', 'right']:
            h_df = ctrl[(ctrl['subject'] == sub) & (ctrl['hemi_label'] == hemi)]
            cv = h_df[h_df['category'] == cat]['geometry_preservation'].values
            if len(cv) and np.isfinite(cv[0]):
                cat_ctrl.append(cv[0])
    cat_ctrl = np.array(cat_ctrl)

    n_sig = 0
    for sub, val in pt_vals:
        t, p = crawford_howell(val, cat_ctrl)
        if p < 0.05:
            n_sig += 1

    pt_mean = np.mean([v for _, v in pt_vals]) if pt_vals else np.nan
    print(f'{cat:<10} {lat:<12} {pt_mean:>7.3f}  {cat_ctrl.mean():>10.3f}  '
          f'{n_sig}/{len(pt_vals)} sig')

TABLE 5: Geometry preservation by resection side

  Left Resection (intact RH) (n=4):
    OTC010: uni=0.808, bil=0.045, diff=-0.764
    OTC017: uni=0.195, bil=0.694, diff=0.498
    OTC021: uni=0.748, bil=0.410, diff=-0.338
    OTC079: uni=0.917, bil=0.594, diff=-0.322
    Mean: uni=0.667, bil=0.436, diff=-0.231 (SD=0.457)
    Direction: 3/4 show uni > bil
    (n<5, Wilcoxon not appropriate)
    Permutation: p=0.4974
    Binomial: p=0.6250

  Right Resection (intact LH) (n=2):
    OTC004: uni=0.760, bil=0.223, diff=-0.537
    OTC008: uni=0.119, bil=-0.082, diff=-0.201
    Mean: uni=0.440, bil=0.071, diff=-0.369 (SD=0.168)
    Direction: 2/2 show uni > bil
    (n<5, Wilcoxon not appropriate)
    Binomial: p=0.5000

TABLE 5b: Same, excluding OTC017

  Left Resection (intact RH), excl OTC017 (n=3):
    OTC010: uni=0.808, bil=0.045, diff=-0.764
    OTC021: uni=0.748, bil=0.410, diff=-0.338
    OTC079: uni=0.917, bil=0.594, diff=-0.322
    Mean diff=-0.475, Direction: 3/3 uni > bil

  Right 

In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# TABLE 7: Bootstrap confidence intervals for geometry preservation
# ═══════════════════════════════════════════════════════════════════════════════

N_BOOT = 100_000
rng_boot = np.random.default_rng(42)

# Rebuild arrays from the lists to avoid name collisions
bilat_arr = np.array(bilat_vals)
uni_arr   = np.array(uni_vals)

def boot_ci(vals, n_boot=N_BOOT, ci=0.95):
    """Bootstrap CI for the mean."""
    vals = np.asarray(vals).ravel()
    means = np.array([np.mean(rng_boot.choice(vals, size=len(vals), replace=True))
                      for _ in range(n_boot)])
    lo = (1 - ci) / 2
    return np.percentile(means, [lo * 100, (1 - lo) * 100])

def boot_ci_diff(a, b, n_boot=N_BOOT, ci=0.95):
    """Bootstrap CI for the mean of paired differences (a - b)."""
    diffs = np.asarray(a).ravel() - np.asarray(b).ravel()
    means = np.array([np.mean(rng_boot.choice(diffs, size=len(diffs), replace=True))
                      for _ in range(n_boot)])
    lo = (1 - ci) / 2
    return np.mean(diffs), np.percentile(means, [lo * 100, (1 - lo) * 100])

# ── OTC: symmetric vs asymmetric ─────────────────────────────────────────────
print('TABLE 7: Bootstrap CIs for geometry preservation')
print(f'  Resamples: {N_BOOT:,}')
print()

# Full sample
m_diff, ci_full = boot_ci_diff(bilat_arr, uni_arr)
ci_bil = boot_ci(bilat_arr)
ci_uni = boot_ci(uni_arr)
print(f'OTC full sample (n={len(bilat_arr)}):')
print(f'  Symmetric M   = {bilat_arr.mean():.3f}  95% CI [{ci_bil[0]:.3f}, {ci_bil[1]:.3f}]')
print(f'  Asymmetric M  = {uni_arr.mean():.3f}  95% CI [{ci_uni[0]:.3f}, {ci_uni[1]:.3f}]')
print(f'  Diff (sym-asym) = {m_diff:.3f}  95% CI [{ci_full[0]:.3f}, {ci_full[1]:.3f}]')
sig = 'significant' if ci_full[0] > 0 or ci_full[1] < 0 else 'not significant'
print(f'  → {sig} (CI {"excludes" if sig == "significant" else "includes"} zero)')
print()

# Excluding OTC017
b_ex = np.array([b for b, u, s in zip(bilat_vals, uni_vals, subs_used) if s != 'OTC017'])
u_ex = np.array([u for b, u, s in zip(bilat_vals, uni_vals, subs_used) if s != 'OTC017'])
m_ex, ci_ex = boot_ci_diff(b_ex, u_ex)
print(f'OTC excluding OTC017 (n={len(b_ex)}):')
print(f'  Diff (sym-asym) = {m_ex:.3f}  95% CI [{ci_ex[0]:.3f}, {ci_ex[1]:.3f}]')
sig_ex = 'significant' if ci_ex[0] > 0 or ci_ex[1] < 0 else 'not significant'
print(f'  → {sig_ex}')
print()

# ── Controls: symmetric vs asymmetric ─────────────────────────────────────────
ctrl_ci = boot_ci(ctrl_diffs)
print(f'Controls (N={len(ctrl_diffs)} hemispheres):')
print(f'  Diff (sym-asym) M = {ctrl_diffs.mean():.3f}  95% CI [{ctrl_ci[0]:.3f}, {ctrl_ci[1]:.3f}]')
sig_ctrl = 'significant' if ctrl_ci[0] > 0 or ctrl_ci[1] < 0 else 'not significant'
print(f'  → {sig_ctrl}')
print()

# ── By resection side ─────────────────────────────────────────────────────────
print('By resection side:')
for side in ['left', 'right']:
    side_b = np.array([b for b, u, s in zip(bilat_vals, uni_vals, subs_used)
                       if otc[otc['subject'] == s]['surgery_side'].iloc[0] == side])
    side_u = np.array([u for b, u, s in zip(bilat_vals, uni_vals, subs_used)
                       if otc[otc['subject'] == s]['surgery_side'].iloc[0] == side])
    if len(side_b) < 2:
        print(f'  {side} resection (n={len(side_b)}): too few for bootstrap')
        continue
    m_s, ci_s = boot_ci_diff(side_b, side_u)
    print(f'  {side} resection (n={len(side_b)}):')
    print(f'    Diff = {m_s:.3f}  95% CI [{ci_s[0]:.3f}, {ci_s[1]:.3f}]')
print()

# ── Per category ──────────────────────────────────────────────────────────────
print('Per-category geometry preservation (OTC):')
for cat in ['face', 'word', 'object', 'house']:
    vals = []
    for sub in sorted(otc['subject'].unique()):
        sub_df = otc[otc['subject'] == sub]
        cv = sub_df[sub_df['category'] == cat]['geometry_preservation'].values
        if len(cv) and np.isfinite(cv[0]):
            vals.append(cv[0])
    vals = np.array(vals)
    ci = boot_ci(vals)
    lat = 'asymmetric' if cat in UNILATERAL else 'symmetric'
    print(f'  {cat:<8} ({lat}): M={vals.mean():.3f}  95% CI [{ci[0]:.3f}, {ci[1]:.3f}]  n={len(vals)}')

TABLE 7: Bootstrap CIs for geometry preservation
  Resamples: 100,000

OTC full sample (n=6):
  Symmetric M   = 0.314  95% CI [0.093, 0.535]
  Asymmetric M  = 0.591  95% CI [0.346, 0.824]
  Diff (sym-asym) = -0.277  95% CI [-0.557, 0.070]
  → not significant (CI includes zero)

OTC excluding OTC017 (n=5):
  Diff (sym-asym) = -0.432  95% CI [-0.628, -0.277]
  → significant

Controls (N=18 hemispheres):
  Diff (sym-asym) M = -0.080  95% CI [-0.267, 0.099]
  → not significant

By resection side:
  left resection (n=4):
    Diff = -0.231  95% CI [-0.653, 0.289]
  right resection (n=2):
    Diff = -0.369  95% CI [-0.537, -0.201]

Per-category geometry preservation (OTC):
  face     (asymmetric): M=0.512  95% CI [0.045, 0.867]  n=6
  word     (asymmetric): M=0.622  95% CI [0.301, 0.859]  n=5
  object   (symmetric): M=0.485  95% CI [0.214, 0.721]  n=6
  house    (symmetric): M=0.143  95% CI [-0.192, 0.473]  n=6
